# RadiNova AI — Chest X-Ray Model Training (DenseNet-121 + Grad-CAM)

**Objective:** Train a high-sensitivity DenseNet-121 binary classifier for Pneumonia detection on the Kaggle Chest X-Ray dataset with 80/10/10 stratified split, evaluate clinical metrics (Recall / Sensitivity as priority), test Grad-CAM heatmap visualization, and export `chest_densenet121.pth` weights for the RadiNova AI dashboard.

> **Disclaimer:** *For educational/research purposes only — not a substitute for professional medical diagnosis.*

### Step 1: Install Dependencies & Setup Environment

In [ ]:
!pip install -q kagglehub torch torchvision scikit-learn pandas pillow opencv-python matplotlib
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Step 2: Download Dataset & Perform 80/10/10 Stratified Re-Split

In [ ]:
import kagglehub
import os, random, csv
from pathlib import Path
import pandas as pd

print("Downloading Kaggle chest-xray-pneumonia dataset...")
dataset_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print(f"Downloaded to: {dataset_path}")

# Find all images and build stratified 80/10/10 split
valid_exts = {".jpeg", ".jpg", ".png", ".JPEG", ".JPG", ".PNG"}
samples = []
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        if Path(f).suffix in valid_exts:
            full_p = os.path.join(root, f)
            if "NORMAL" in full_p.upper():
                samples.append((full_p, "NORMAL"))
            elif "PNEUMONIA" in full_p.upper():
                samples.append((full_p, "PNEUMONIA"))

print(f"Total images scanned: {len(samples)}")

buckets = {}
for p, l in samples:
    buckets.setdefault(l, []).append((p, l))

rng = random.Random(42)
manifest = []
for lbl, items in buckets.items():
    rng.shuffle(items)
    n = len(items)
    n_tr = int(n * 0.80)
    n_v = int(n * 0.10)
    for p, _ in items[:n_tr]: manifest.append({"filepath": p, "label": lbl, "split": "train"})
    for p, _ in items[n_tr:n_tr+n_v]: manifest.append({"filepath": p, "label": lbl, "split": "val"})
    for p, _ in items[n_tr+n_v:]: manifest.append({"filepath": p, "label": lbl, "split": "test"})

df_manifest = pd.DataFrame(manifest)
df_manifest.to_csv("chest_xray_manifest.csv", index=False)
print("Manifest saved: chest_xray_manifest.csv")
print(df_manifest.groupby(["split", "label"]).size())

### Step 3: Dataset Class & Model Definition (DenseNet-121)

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import numpy as np

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
CLASS_TO_IDX = {"NORMAL": 0, "PNEUMONIA": 1}
CLASSES = ["NORMAL", "PNEUMONIA"]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

class XRayDataset(Dataset):
    def __init__(self, df, split, tf):
        self.data = df[df["split"] == split].reset_index(drop=True)
        self.tf = tf
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        return self.tf(img), CLASS_TO_IDX[row["label"]]

train_ds = XRayDataset(df_manifest, "train", train_tf)
val_ds = XRayDataset(df_manifest, "val", val_tf)
test_ds = XRayDataset(df_manifest, "test", val_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

# Build DenseNet-121
model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.classifier.in_features, 2)
)
model = model.to(device)
print("DenseNet-121 initialized with custom binary classifier head.")

### Step 4: Model Training & Evaluation

In [ ]:
def eval_metrics(model, loader):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outs = model(imgs)
            _, preds = torch.max(outs, 1)
            all_p.extend(preds.cpu().numpy())
            all_l.extend(lbls.cpu().numpy())
    y_true, y_pred = np.array(all_l), np.array(all_p)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    acc = (tp + tn) / max(len(y_true), 1)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * (prec * rec) / max(prec + rec, 1e-6)
    return acc, prec, rec, f1, (tn, fp, fn, tp)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)

best_f1 = 0.0
epochs = 8

for epoch in range(1, epochs + 1):
    model.train()
    r_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        r_loss += loss.item() * imgs.size(0)
    scheduler.step()
    
    val_acc, val_prec, val_rec, val_f1, _ = eval_metrics(model, val_loader)
    print(f"Epoch [{epoch}/{epochs}] Train Loss: {r_loss/len(train_ds):.4f} | Val Acc: {val_acc*100:.1f}% Rec (Sens): {val_rec*100:.1f}% F1: {val_f1:.4f}")
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({"model_state_dict": model.state_dict(), "classes": CLASSES}, "chest_densenet121.pth")
        print(f"  >>> Checkpoint saved (F1: {best_f1:.4f}) -> chest_densenet121.pth")

### Step 5: Test Set Performance & Medical Metrics (Recall / Sensitivity Priority)

In [ ]:
checkpoint = torch.load("chest_densenet121.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
acc, prec, rec, f1, (tn, fp, fn, tp) = eval_metrics(model, test_loader)

print("="*50)
print("   RADINOVA AI — FINAL TEST EVALUATION RESULTS")
print("="*50)
print(f"Accuracy:                 {acc*100:.2f}%")
print(f"Precision (PPV):          {prec*100:.2f}%")
print(f"Recall / Sensitivity:     {rec*100:.2f}%  <-- CLINICAL PRIORITY METRIC")
print(f"Specificity (TNR):        {tn/(tn+fp)*100:.2f}%")
print(f"F1-Score:                 {f1:.4f}")
print("-"*50)
print(f"Confusion Matrix:")
print(f"                  Pred NORMAL      Pred PNEUMONIA")
print(f"Actual NORMAL         {tn:<16} {fp:<16}")
print(f"Actual PNEUMONIA      {fn:<16} {tp:<16}")
print("="*50)

### Step 6: Download Trained Model Weights (`chest_densenet121.pth`)

In [ ]:
from google.colab import files
files.download("chest_densenet121.pth")
print("Place downloaded 'chest_densenet121.pth' inside RadiNova AI at 'model/weights/chest_densenet121.pth'")